In [6]:
import os
import pyzed.sl as sl
import cv2
import numpy as np
import threading
import time
import signal

import time
import threading
import matplotlib.pyplot as plt

### Record videos

In [25]:
zed_list = []
current_images = []
current_timestamps = []
thread_list = []
stop_signal = False




def signal_handler(signal, frame):
    global stop_signal
    stop_signal=True
    time.sleep(0.5)
    exit()

def grab_run(index):
    global stop_signal
    global zed_list
    global current_timestamps
    global current_images
    global name_list

    

    runtime = sl.RuntimeParameters()
    while not stop_signal:

        err = zed_list[index].grab(runtime)
        if err == sl.ERROR_CODE.SUCCESS:

            zed_list[index].retrieve_image(current_images[index], sl.VIEW.LEFT)
            current_timestamps[index] = zed_list[index].get_timestamp(sl.TIME_REFERENCE.CURRENT).data_ns #time of function call


        barrier.wait()

        time.sleep(0.001) #0.1ms
    zed_list[index].disable_recording()
    zed_list[index].close()

def main():
    global stop_signal
    global zed_list
    global current_images
    global current_timestamps
    global thread_list
    global name_list
    global barrier
    signal.signal(signal.SIGINT, signal_handler)

    output_file_names = ['multicam_1.svo2','multicam_2.svo2']

    print("Running...")
    init = sl.InitParameters()
    init.camera_resolution = sl.RESOLUTION.HD1080
    init.camera_fps = 60  # The framerate is lowered to avoid any USB3 bandwidth issues
    # init.async_image_retrieval = True


    #List and open cameras
    name_list = []
    last_ts_list = []
    cameras = sl.Camera.get_device_list()

    index = 0
    for cam in cameras:
        init.set_from_serial_number(cam.serial_number) # define input source
        name_list.append(f"ZED_{cam.serial_number}")

        print(f"Opening {name_list[index]}")
        zed_list.append(sl.Camera())
        current_images.append(sl.Mat())
        current_timestamps.append(0)
        last_ts_list.append(0)

        # open cameras
        status = zed_list[index].open(init)
        if status != sl.ERROR_CODE.SUCCESS:
            print(repr(status))
            zed_list[index].close()

        # set recording params
        recording_param = sl.RecordingParameters(
            os.path.join('./', output_file_names[index]),
            sl.SVO_COMPRESSION_MODE.H264) # Enable recording with the filename specified in argument
        err = zed_list[index].enable_recording(recording_param)
        index = index +1

        if err != sl.ERROR_CODE.SUCCESS:
            print("Recording ZED : ", err)
            exit(1)

    num_threads = len(zed_list)
    barrier = threading.Barrier(num_threads)

    #Start camera threads
    for index in range(0, len(zed_list)):
        if zed_list[index].is_opened():
            thread_list.append(threading.Thread(target=grab_run, args=(index,)))
            thread_list[index].start()
    
    #Display camera images
    key = ''
    while key != 113:  # for 'q' key
        for cam_index in range(0, len(zed_list)):
            if zed_list[cam_index].is_opened():

                # if there is a new frame 
                if (current_timestamps[cam_index] >= last_ts_list[cam_index]):
                    img = current_images[cam_index].get_data()

                    img = cv2.circle(img,(960,540),10,(0,0,255),3)
                    
                    cv2.imshow(name_list[cam_index], cv2.resize(img,(800,600)))

                    # set the last timestamp for each cam to the CURRENT_ts one of each cam 
                    last_ts_list[cam_index] = current_timestamps[cam_index]

        key = cv2.waitKey(10)
    cv2.destroyAllWindows()

    #Stop the threads
    stop_signal = True
    for index in range(0, len(thread_list)):
        thread_list[index].join()
    print("\nFINISH")

if __name__ == "__main__":
    main()

Running...
[2025-02-13 14:19:09 UTC][ZED][INFO] Logging level INFO
Opening ZED_43957242
[2025-02-13 14:19:09 UTC][ZED][INFO] Logging level INFO
[2025-02-13 14:19:10 UTC][ZED][INFO] Logging level INFO
[2025-02-13 14:19:10 UTC][ZED][INFO] Logging level INFO
[2025-02-13 14:19:10 UTC][ZED][INFO] Logging level INFO
[2025-02-13 14:19:10 UTC][ZED][INFO] [Init]  Depth mode: PERFORMANCE
[2025-02-13 14:19:15 UTC][ZED][INFO] [Init]  Camera FW version: 2001
[2025-02-13 14:19:15 UTC][ZED][INFO] [Init]  Video mode: HD1080@60
[2025-02-13 14:19:15 UTC][ZED][INFO] [Init]  Serial Number: S/N 43957242
[2025-02-13 14:19:16 UTC][ZED][INFO] [Init]  Notice: The recording is using SVO version 2, enabled by default starting from SDK version 4.1. To revert to the original SVO version, set the environment variable "ZED_SDK_SVO_VERSION" to 1
Opening in BLOCKING MODE 
Opening ZED_43916681


NvMMLiteOpen : Block : BlockType = 4 
===== NvVideo: NVENC =====
NvMMLiteBlockCreate : Block : BlockType = 4 
H264: Profile = 77, Level = 50 
NVMEDIA: Need to set EMC bandwidth : 5744000 


[2025-02-13 14:19:16 UTC][ZED][INFO] Logging level INFO
[2025-02-13 14:19:16 UTC][ZED][INFO] Logging level INFO
[2025-02-13 14:19:16 UTC][ZED][INFO] Logging level INFO
[2025-02-13 14:19:16 UTC][ZED][INFO] [Init]  Depth mode: PERFORMANCE
[2025-02-13 14:19:20 UTC][ZED][INFO] [Init]  Camera FW version: 2001
[2025-02-13 14:19:20 UTC][ZED][INFO] [Init]  Video mode: HD1080@60
[2025-02-13 14:19:20 UTC][ZED][INFO] [Init]  Serial Number: S/N 43916681
[2025-02-13 14:19:20 UTC][ZED][WARNING] [Init]  Self-calibration failed. Point the camera towards a more textured and brighter area. Avoid objects closer than 1 meter (Error code: 0x01) 
[2025-02-13 14:19:21 UTC][ZED][INFO] [Init]  Notice: The recording is using SVO version 2, enabled by default starting from SDK version 4.1. To revert to the original SVO version, set the environment variable "ZED_SDK_SVO_VERSION" to 1
Opening in BLOCKING MODE 


NvMMLiteOpen : Block : BlockType = 4 
===== NvVideo: NVENC =====
NvMMLiteBlockCreate : Block : BlockType = 4 
H264: Profile = 77, Level = 50 
NVMEDIA: Need to set EMC bandwidth : 5744000 
NvVideo: bBlitMode is set to TRUE 
NvVideo: bBlitMode is set to TRUE 



FINISH


### Playback

In [4]:
import os
import cv2
import pyzed.sl as sl
from Util.util import progress_bar

data = {
    'fn_timestamp':[],
    'img_timestamp':[],
    'imgs':[]
}

kepek = []


def main():

    input_folder = "./"
    input_file_name = "multicam_2.svo2"
    filepath = os.path.join(input_folder, input_file_name)

    input_type = sl.InputType()
    input_type.set_from_svo_file(filepath)  # Set init parameter to run from the .svo
    init = sl.InitParameters(input_t=input_type, svo_real_time_mode=False)

    cam = sl.Camera()
    status = cam.open(init)
    if status != sl.ERROR_CODE.SUCCESS:  # Ensure the camera opened succesfully
        print("Camera Open", status, "Exit program.")
        exit(1)

    # Set a maximum resolution, for visualisation confort
    resolution = cam.get_camera_information().camera_configuration.resolution
    low_resolution = sl.Resolution(
        min(720, resolution.width), min(404, resolution.height)
    )
    svo_image = sl.Mat(
        min(720, resolution.width),
        min(404, resolution.height),
        sl.MAT_TYPE.U8_C4,
        sl.MEM.CPU,
    )

    runtime = sl.RuntimeParameters()
    nb_frames = cam.get_svo_number_of_frames()

    print("[Info] SVO contains ", nb_frames, " frames")

    key = ''
    cv2.namedWindow('faszkivan')
    while True:
        key = cv2.waitKey()
        
        if key == 45:
                cam.set_svo_position(svo_position-10)

        if key == 43:

        

            err = cam.grab(runtime)
            if err == sl.ERROR_CODE.SUCCESS:
                cam.retrieve_image(svo_image, sl.VIEW.LEFT, sl.MEM.CPU, low_resolution)
                fn_timestamp = cam.get_timestamp(sl.TIME_REFERENCE.CURRENT).data_ns
                img_timestamp = cam.get_timestamp(sl.TIME_REFERENCE.CURRENT).data_ns
                fps = cam.get_current_fps()
                svo_position = cam.get_svo_position()

                text = f'fn: {fn_timestamp} \n img: {img_timestamp} \n fps: {fps}'
                img = svo_image.get_data()
                # cv2.putText(img, text, (30,30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1, cv2.LINE_AA)
                cv2.imshow("View", img)  # dislay both images to cv2
                print(text) 
            else:
                print("Grab ZED : ", err)
                break

        if key == 113:
            break

           


    cv2.destroyAllWindows()
    cam.close()


if __name__ == "__main__":

    main()


[2025-02-13 15:50:34 UTC][ZED][INFO] Logging level INFO
[Info] SVO contains  637  frames
[2025-02-13 15:50:34 UTC][ZED][INFO] Logging level INFO
[2025-02-13 15:50:34 UTC][ZED][INFO] Logging level INFO
Opening in BLOCKING MODE 
[2025-02-13 15:50:34 UTC][ZED][INFO] [Init]  Depth mode: PERFORMANCE
[2025-02-13 15:50:34 UTC][ZED][INFO] [Init]  Serial Number: S/N 43916681
[2025-02-13 15:50:35 UTC][ZED][WARNING] [Init]  Self-calibration failed. Point the camera towards a more textured and brighter area. Avoid objects closer than 1 meter (Error code: 0x01) 


NvMMLiteOpen : Block : BlockType = 261 
NvMMLiteBlockCreate : Block : BlockType = 261 


fn: 1739461838154744923 
 img: 1739461838154751995 
 fps: 0.0
fn: 1739461839958046295 
 img: 1739461839958057591 
 fps: 0.5534034371376038
fn: 1739461840458281546 
 img: 1739461840458289642 
 fps: 0.8669267296791077
fn: 1739461840485768993 
 img: 1739461840485777761 
 fps: 1.281503677368164
fn: 1739461840509791673 
 img: 1739461840509801753 
 fps: 1.6884762048721313
fn: 1739461840539718118 
 img: 1739461840539724710 
 fps: 2.085070848464966
fn: 1739461840562143216 
 img: 1739461840562150544 
 fps: 2.4691357612609863
fn: 1739461840593696076 
 img: 1739461840593702252 
 fps: 2.8466856479644775
fn: 1739461840625766508 
 img: 1739461840625772236 
 fps: 3.2128515243530273
fn: 1739461840657377609 
 img: 1739461840657384041 
 fps: 3.5714285373687744
fn: 1739461840691079128 
 img: 1739461840691086040 
 fps: 3.9184954166412354
fn: 1739461840718632847 
 img: 1739461840718638767 
 fps: 12.903225898742676
fn: 1739461840841724258 
 img: 1739461840841730818 
 fps: 32.78688430786133
fn: 1739461840857

In [ ]:
cam1_fn = np.array([1739461432213468817,
                    1739461674466071676,
                    1739461707246427225,
                    1739461744323333921,
                    1739461771631301163,
                    1739461801440064082])/1000000

cam1_img = np.array([1739461432213478993,
                     1739461674466082812,
                     1739461707246435801,
                     1739461744323342945,
                     1739461771631310379,
                     1739461801440074962])/1000000

cam2_fn = np.array([1739461856587684348,
                    1739461892295466039,
                    1739461925437439188,
                    1739461945978010036,
                    1739461966833438624,
                    1739461996298820297])/1000000

cam2_img = np.array([1739461856587692060,
                     1739461892295473271,
                     1739461925437444980,
                     1739461945978020596,
                     1739461966833445088,
                     1739461996298820297])/1000000

In [30]:
(1739461432213468817-1739461856587684348)/1000000/1000/60

-7.072903592183334

In [26]:
cam1_fn[0]

1.7394614322134688e+19

In [19]:
np.array(cam1_img-cam2_img)/1000/60

array([-7.07290355, -3.63048984, -3.63651682, -3.3609113 , -3.25336891,
       -3.24764576])